# Agentic RAG

This notebook introduces the next layer above the current RAG pipeline: adding decision-making around retrieval and generation.

The current pipeline is mostly linear:

```text
question -> retrieve chunks -> build context -> generate answer
```

Agentic RAG turns that into a controlled workflow:

```text
question
  -> validate whether the question belongs to the corpus domain
  -> retrieve candidate chunks
  -> grade whether retrieved chunks are useful
  -> rewrite the query if retrieval is weak
  -> retry retrieval within a limit
  -> generate an answer only when there is enough evidence
```

The goal is not to make the system more complicated. The goal is to prevent the system from confidently answering with bad context.

## 0. What Problem Are We Solving?

A search engine always returns its best available results. That does not mean the results are actually relevant.

With a small corpus, this is especially dangerous:

```text
User: What is photosynthesis?
Retriever: Here are the closest chunks about DiffusionGemma.
LLM: Based on the provided papers, photosynthesis relates to DiffusionGemma...
```

That is a retrieval-control failure. The generator is doing what it was asked to do: answer from the supplied context. The missing layer is a decision system that asks whether retrieval should happen, whether retrieved chunks are good, and whether the system should refuse.

## 1. Traditional RAG vs Agentic RAG

| Concern | Traditional RAG | Agentic RAG |
|---|---|---|
| Query validation | Usually none | Validate scope before retrieval |
| Retrieval | Always retrieve | Retrieve only if the query passes guardrails |
| Relevance check | Trust top-k | Grade retrieved chunks |
| Bad retrieval | Still generate | Rewrite query or refuse |
| Control flow | Linear | Branching graph |
| Debuggability | Hard to inspect | Reasoning steps are explicit |

The practical shift is this: the RAG system stops being a single pipeline and becomes a small workflow.

## 2. Mental Model: A Graph, Not a Chain

A chain always moves forward in one fixed order.

A graph can branch:

```text
START
  -> guardrail
       -> reject if out of scope
       -> retrieve if in scope
  -> grade retrieved chunks
       -> generate if relevant
       -> rewrite query if weak
  -> retry retrieve
  -> END
```

LangGraph is a framework for building this kind of stateful graph. In this notebook, we first implement the ideas with plain Python so the control flow is obvious. Later, the same nodes can be moved into a LangGraph implementation.

## 3. Core Concepts

**Agent state** is the shared object passed between nodes. It stores the question, current query, retrieved chunks, answer, retrieval attempts, and reasoning trace.

**Nodes** are small functions. Each node does one job: validate, retrieve, grade, rewrite, or generate.

**Edges** decide where the workflow goes next.

**Conditional edges** are the important part. Example: if the guardrail score is low, go to refusal. Otherwise, go to retrieval.

**Reasoning steps** are not chain-of-thought. They are system-level audit events: `validated query`, `retrieved 5 chunks`, `rewrote query`, `generated answer`.

## 4. Setup

This notebook can run in two modes:

- Local concept mode: uses lightweight Python functions to demonstrate graph control flow.
- Project API mode: calls your running FastAPI service for health checks and baseline RAG comparison.

Start the services before running API cells:

```bash
docker compose up -d
```

In [6]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import requests

print(f"Python: {sys.version.split()[0]}")

current_dir = Path.cwd()
if (current_dir / "docker-compose.yml").exists():
    project_root = current_dir
elif (current_dir.parent / "docker-compose.yml").exists():
    project_root = current_dir.parent
else:
    project_root = current_dir.parent.parent

print(f"Project root: {project_root}")

env_file = project_root / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key, value)
    print("Loaded .env values into this notebook process")
else:
    print("No .env file found. API cells may still work if services use Docker env vars.")

API_BASE = os.getenv("API_BASE", "http://localhost:8000")
REQUEST_TIMEOUT = 300
print(f"API_BASE: {API_BASE}")

Python: 3.11.13
Project root: /Users/shishirrijal/Desktop/Projects/arxiv-rag-curator
Loaded .env values into this notebook process
API_BASE: http://localhost:8000


## 5. Baseline Health Check

Before building agentic behavior, confirm the existing system is reachable.

In [7]:
def get_health() -> dict[str, Any]:
    response = requests.get(f"{API_BASE}/health", timeout=10)
    response.raise_for_status()
    return response.json()


try:
    health = get_health()
    print(json.dumps(health, indent=2))
except Exception as exc:
    print(f"API health check failed: {exc}")
    print("Continue with local concept cells, or start services with: docker compose up -d")

{
  "status": "healthy",
  "version": "0.4.0",
  "services": {
    "postgresql": {
      "status": "healthy"
    },
    "opensearch": {
      "status": "healthy",
      "cluster_status": "green"
    },
    "ollama": {
      "status": "healthy",
      "model": "llama3.2:1b",
      "model_ready": true,
      "available_models": [
        "llama3.2:1b"
      ]
    },
    "redis": {
      "status": "healthy",
      "ttl_seconds": 86400
    },
    "langfuse": {
      "status": "healthy",
      "host": "https://us.cloud.langfuse.com"
    },
    "embeddings": {
      "status": "enabled",
      "reason": null,
      "fallback": "bm25"
    }
  }
}


## 6. The State Object

A graph needs memory. Not long-term memory, just workflow memory.

For this project, useful state includes:

- original question
- current query being searched
- guardrail score
- retrieved hits
- whether documents are relevant
- answer
- retrieval attempts
- rewritten query
- reasoning steps

In [16]:
@dataclass
class AgentConfig:
    guardrail_threshold: int = 40
    max_retrieval_attempts: int = 2
    top_k: int = 5
    use_hybrid: bool = True


@dataclass
class AgentState:
    question: str
    current_query: str
    guardrail_score: Optional[int] = None
    retrieved_hits: list[dict[str, Any]] = field(default_factory=list)
    documents_relevant: bool = False
    answer: Optional[str] = None
    retrieval_attempts: int = 0
    rewritten_query: Optional[str] = None
    reasoning_steps: list[str] = field(default_factory=list)
    rejected: bool = False


state = AgentState(
    question="What is the transformer architecture?",
    current_query="What is the transformer architecture?",
)
state

AgentState(question='What is the transformer architecture?', current_query='What is the transformer architecture?', guardrail_score=None, retrieved_hits=[], documents_relevant=False, answer=None, retrieval_attempts=0, rewritten_query=None, reasoning_steps=[], rejected=False)

## 7. Node 1: Guardrail Validation

The guardrail decides whether the question belongs to the system domain before retrieval.

In a production version, this is usually an LLM call that returns structured JSON, for example:

```json
{
  "score": 85,
  "reason": "The question is about neural network architecture."
}
```

For this learning notebook, the first implementation is deterministic and lightweight. This is not the final production guardrail. It exists so we can understand the control flow without waiting on an LLM for every cell.

In [17]:
DOMAIN_TERMS = {
    "attention", "transformer", "bert", "gpt", "llm", "language", "model",
    "neural", "network", "embedding", "retrieval", "rag", "diffusion",
    "machine", "learning", "nlp", "classification", "training", "inference",
}


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def guardrail_node(state: AgentState, config: AgentConfig) -> AgentState:
    terms = tokenize(state.question)
    overlap = terms & DOMAIN_TERMS
    score = min(100, 25 + len(overlap) * 25) if overlap else 10
    state.guardrail_score = score

    if score < config.guardrail_threshold:
        state.rejected = True
        state.answer = (
            "I could not find a reason to treat this as a question about the "
            "indexed AI/ML research corpus. Ask about papers, models, retrieval, "
            "or related research topics."
        )
        state.reasoning_steps.append(f"Rejected by guardrail with score {score}/100")
    else:
        state.reasoning_steps.append(f"Accepted by guardrail with score {score}/100")

    return state


for q in ["What is photosynthesis?", "What is the transformer architecture? Also explain neural network and llm with reference to it.", "How does retrieval augmented generation work?"]:
    s = AgentState(question=q, current_query=q)
    guardrail_node(s, AgentConfig())
    print(q, "=>", s.guardrail_score, s.rejected, s.reasoning_steps[-1])

What is photosynthesis? => 10 True Rejected by guardrail with score 10/100
What is the transformer architecture? Also explain neural network and llm with reference to it. => 100 False Accepted by guardrail with score 100/100
How does retrieval augmented generation work? => 50 False Accepted by guardrail with score 50/100


## 8. Conditional Edge: Continue or Reject

A conditional edge is just a decision function.

```text
if rejected:
    END
else:
    retrieve
```

LangGraph formalizes this pattern. Plain Python can show the idea directly.

In [18]:
def after_guardrail(state: AgentState) -> str:
    return "end" if state.rejected else "retrieve"


for q in ["What is photosynthesis?", "Explain self-attentionin transformer neural network."]:
    s = guardrail_node(AgentState(question=q, current_query=q), AgentConfig())
    print(q, "-> next node:", after_guardrail(s))

What is photosynthesis? -> next node: end
Explain self-attentionin transformer neural network. -> next node: retrieve


## 9. Node 2: Retrieval

Retrieval is a tool call. The graph does not need to know OpenSearch internals. It only needs a node that can turn a query into candidate chunks.

For now, this notebook calls your existing `/api/v1/hybrid-search/` endpoint if the API is running.

In [19]:
def retrieve_from_api(query: str, *, use_hybrid: bool = True, top_k: int = 5) -> list[dict[str, Any]]:
    response = requests.post(
        f"{API_BASE}/api/v1/hybrid-search/",
        json={"query": query, "use_hybrid": use_hybrid, "page_size": top_k},
        timeout=30,
    )
    response.raise_for_status()
    return response.json().get("hits", [])


def retrieve_node(state: AgentState, config: AgentConfig) -> AgentState:
    state.retrieval_attempts += 1
    try:
        state.retrieved_hits = retrieve_from_api(
            state.current_query,
            use_hybrid=config.use_hybrid,
            top_k=config.top_k,
        )
        state.reasoning_steps.append(
            f"Retrieved {len(state.retrieved_hits)} chunks for query: {state.current_query!r}"
        )
    except Exception as exc:
        state.retrieved_hits = []
        state.reasoning_steps.append(f"Retrieval failed: {exc}")
    return state


s = AgentState(
    question="What is the transformer architecture?",
    current_query="What is the transformer architecture?",
)
guardrail_node(s, AgentConfig())
if not s.rejected:
    retrieve_node(s, AgentConfig(top_k=3))

print("attempts:", s.retrieval_attempts)
print("hits:", len(s.retrieved_hits))
for hit in s.retrieved_hits[:2]:
    print("-", hit.get("title"), "| score:", hit.get("score"))

attempts: 1
hits: 3
- Attention Is All You Need | score: 0.5
- How Transparent is DiffusionGemma? | score: 0.5


## 10. Node 3: Document Grading

Document grading answers this question:

```text
Do the retrieved chunks contain enough evidence to answer the user question?
```

This is different from search ranking. Search asks, `Which chunks are closest?` Grading asks, `Are these chunks good enough?`

Production versions often use an LLM with structured output. This notebook uses a simple overlap grader to make the control flow visible.

In [20]:
BASIC_STOPWORDS = {
    "the", "and", "for", "with", "what", "why", "how", "are", "is", "to", "of",
    "in", "a", "an", "it", "this", "that", "about", "explain", "tell", "me",
}


def content_terms(text: str) -> set[str]:
    return {t for t in tokenize(text) if len(t) > 2 and t not in BASIC_STOPWORDS}


def grade_documents_node(state: AgentState, config: AgentConfig) -> AgentState:
    if not state.retrieved_hits:
        state.documents_relevant = False
        state.reasoning_steps.append("Document grading: no retrieved chunks")
        return state

    query_terms = content_terms(state.question)
    relevant_hits = 0

    for hit in state.retrieved_hits:
        text = " ".join([
            hit.get("title", ""),
            hit.get("abstract", ""),
            hit.get("chunk_text", "")[:1200],
        ])
        overlap = query_terms & content_terms(text)
        if len(overlap) >= 2:
            relevant_hits += 1

    state.documents_relevant = relevant_hits > 0
    state.reasoning_steps.append(
        f"Document grading: {relevant_hits}/{len(state.retrieved_hits)} chunks look relevant"
    )
    return state


grade_documents_node(s, AgentConfig())
print("documents_relevant:", s.documents_relevant)
print("last step:", s.reasoning_steps[-1])

documents_relevant: True
last step: Document grading: 2/3 chunks look relevant


## 11. Conditional Edge: Generate or Rewrite

After grading, the graph needs another decision:

```text
if documents are relevant:
    generate answer
elif retrieval attempts remain:
    rewrite query
else:
    refuse because evidence is insufficient
```

In [21]:
def after_grading(state: AgentState, config: AgentConfig) -> str:
    if state.documents_relevant:
        return "generate"
    if state.retrieval_attempts < config.max_retrieval_attempts:
        return "rewrite"
    return "insufficient_evidence"


print(after_grading(s, AgentConfig()))

generate


## 12. Node 4: Query Rewriting

Query rewriting helps when the user asks a vague but in-scope question.

Example:

```text
Original: Tell me about ML stuff
Rewritten: machine learning methods neural networks training retrieval language models
```

A production implementation should use an LLM and include the domain/corpus description in the prompt. This notebook uses a simple deterministic rewrite to demonstrate where the step fits.

In [22]:
def rewrite_query_node(state: AgentState, config: AgentConfig) -> AgentState:
    original = state.current_query
    terms = content_terms(original)

    if len(terms) <= 3:
        rewritten = f"{original} machine learning neural networks transformer attention retrieval"
    else:
        rewritten = f"{original} research paper method results"

    state.current_query = rewritten
    state.rewritten_query = rewritten
    state.reasoning_steps.append(f"Rewrote query from {original!r} to {rewritten!r}")
    return state


rewrite_demo = AgentState(question="Tell me about ML stuff", current_query="Tell me about ML stuff")
rewrite_query_node(rewrite_demo, AgentConfig())
rewrite_demo.rewritten_query

'Tell me about ML stuff machine learning neural networks transformer attention retrieval'

## 13. Node 5: Generation

Generation should happen only after the graph decides the context is usable.

For this notebook, we call your existing `/api/v1/ask/` endpoint for the final answer. Later, the source implementation can reuse lower-level services directly instead of calling the HTTP API from inside itself.

In [23]:
def ask_current_rag(question: str, *, use_hybrid: bool = True) -> dict[str, Any]:
    response = requests.post(
        f"{API_BASE}/api/v1/ask/",
        json={"question": question, "use_hybrid": use_hybrid},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def generate_node(state: AgentState, config: AgentConfig) -> AgentState:
    try:
        result = ask_current_rag(state.current_query, use_hybrid=config.use_hybrid)
        state.answer = result.get("answer")
        state.reasoning_steps.append(
            f"Generated answer with existing RAG endpoint; cached={result.get('cached')}"
        )
    except Exception as exc:
        state.answer = f"Generation failed: {exc}"
        state.reasoning_steps.append(f"Generation failed: {exc}")
    return state

## 14. Putting the Graph Together in Plain Python

This function is intentionally explicit. It shows the same branching that a LangGraph workflow would encode with nodes and conditional edges.

In [26]:
def run_agentic_rag_prototype(question: str, config: Optional[AgentConfig] = None) -> AgentState:
    config = config or AgentConfig()
    state = AgentState(question=question, current_query=question)

    guardrail_node(state, config)
    if after_guardrail(state) == "end":
        return state

    while True:
        retrieve_node(state, config)
        grade_documents_node(state, config)

        next_step = after_grading(state, config)
        if next_step == "generate":
            generate_node(state, config)
            return state

        if next_step == "rewrite":
            rewrite_query_node(state, config)
            continue

        state.answer = "I could not find enough relevant evidence in the indexed papers to answer this."
        state.reasoning_steps.append("Stopped because retrieved evidence was insufficient")
        return state

## 15. Scenario A: Out-of-Scope Question

The key behavior: reject before retrieval. This avoids the common failure where unrelated questions get answered from unrelated research chunks.

In [27]:
result = run_agentic_rag_prototype("What is photosynthesis?")

print("Rejected:", result.rejected)
print("Retrieval attempts:", result.retrieval_attempts)
print("Answer:", result.answer)
print("Reasoning:")
for step in result.reasoning_steps:
    print("-", step)

Rejected: True
Retrieval attempts: 0
Answer: I could not find a reason to treat this as a question about the indexed AI/ML research corpus. Ask about papers, models, retrieval, or related research topics.
Reasoning:
- Rejected by guardrail with score 10/100


## 16. Scenario B: In-Scope Research Question

This should pass guardrails, retrieve documents, grade them, and then generate.

In [28]:
result = run_agentic_rag_prototype("What is the transformer architecture?")

print("Rejected:", result.rejected)
print("Retrieval attempts:", result.retrieval_attempts)
print("Rewritten query:", result.rewritten_query)
print("Answer preview:", (result.answer or "")[:500])
print("Reasoning:")
for step in result.reasoning_steps:
    print("-", step)

Rejected: False
Retrieval attempts: 1
Rewritten query: None
Answer preview: The transformer architecture is a type of neural network that uses self-attention mechanisms to process sequential data, such as text or speech. It consists of an encoder and a decoder, with each layer performing a series of transformations on the input sequence [1]. The encoder maps the input sequence into a higher-dimensional representation, while the decoder generates an output sequence one element at a time.

In the context of the provided papers, the transformer architecture is used to achi
Reasoning:
- Accepted by guardrail with score 50/100
- Retrieved 5 chunks for query: 'What is the transformer architecture?'
- Document grading: 4/5 chunks look relevant
- Generated answer with existing RAG endpoint; cached=False


## 17. Scenario C: Vague But In-Scope Question

This is where query rewriting can help. A vague query may pass guardrails but retrieve weak documents. The graph gets one or two chances to improve the search query.

In [31]:
result = run_agentic_rag_prototype("Tell me about ML stuff inlcuding gpt")

print("Rejected:", result.rejected)
print("Retrieval attempts:", result.retrieval_attempts)
print("Rewritten query:", result.rewritten_query)
print("Answer preview:", (result.answer or "")[:500])
print("Reasoning:")
for step in result.reasoning_steps:
    print("-", step)

Rejected: False
Retrieval attempts: 2
Rewritten query: Tell me about ML stuff inlcuding gpt machine learning neural networks transformer attention retrieval
Answer preview: I could not find enough relevant evidence in the indexed papers to answer this.
Reasoning:
- Accepted by guardrail with score 50/100
- Retrieved 5 chunks for query: 'Tell me about ML stuff inlcuding gpt'
- Document grading: 0/5 chunks look relevant
- Rewrote query from 'Tell me about ML stuff inlcuding gpt' to 'Tell me about ML stuff inlcuding gpt machine learning neural networks transformer attention retrieval'
- Retrieved 5 chunks for query: 'Tell me about ML stuff inlcuding gpt machine learning neural networks transformer attention retrieval'
- Document grading: 0/5 chunks look relevant
- Stopped because retrieved evidence was insufficient


## 18. Why Use LangGraph Later?

The plain Python prototype is useful for learning, but it is not ideal for production.

LangGraph gives us:

- named nodes
- explicit edges
- conditional routing
- state schema
- easier debugging
- better extension points

The source implementation can map the same concepts like this:

```text
AgentState
  question
  current_query
  retrieved_hits
  answer
  reasoning_steps

Nodes
  guardrail
  retrieve
  grade_documents
  rewrite_query
  generate_answer
  refuse

Conditional edges
  after_guardrail
  after_grading
```

## 19. What Should Be LLM-Based in the Real Implementation?

The notebook used simple deterministic logic to make control flow easy to inspect. In source code, several pieces should become proper LLM calls with structured outputs:

- Guardrail validation: score 0-100 and a short reason.
- Document grading: relevant or not relevant, ideally per chunk.
- Query rewriting: improve retrieval without changing the user's intent.

Important production rule: ask for structured JSON and validate it. Do not parse free-form prose if the result controls workflow routing.

## 20. How This Fits the Current Project

The current project already has:

- paper ingestion
- OpenSearch BM25/hybrid search
- chunk indexing
- complete RAG generation
- Redis cache
- Langfuse tracing

The next source-code phase should add a parallel agentic path, not replace the existing one immediately.

Recommended API shape:

```text
POST /api/v1/ask-agentic/
```

Recommended response additions:

```json
{
  "answer": "...",
  "sources": [],
  "reasoning_steps": [],
  "retrieval_attempts": 1,
  "rewritten_query": null,
  "guardrail_score": 85
}
```

Keeping both endpoints makes comparison easy:

```text
/ask/          -> normal RAG
/ask-agentic/  -> graph-controlled RAG
```

## 21. Summary

The main idea is control.

Agentic RAG adds a decision layer around retrieval and generation:

- validate the query before searching
- retrieve candidate chunks
- grade retrieved chunks before trusting them
- rewrite weak queries and retry within a limit
- refuse when evidence is insufficient
- expose reasoning steps for debugging

This directly improves a common RAG failure mode: answering unrelated questions with whatever chunks OpenSearch happened to return.